# Learning Objectives

* Build a small Pegasus workflow from Python wrapper scripts and an Apptainer container.
* Register remote inputs (Project Gutenberg URLs) in the Replica Catalog and let Pegasus stage them — no download job.
* See fan-out (one job per book) and fan-in (one merge job) in a workflow DAG.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the chart and CSV the run produces.

# wordfreq: Word Frequencies in Project Gutenberg Books

Counts word frequencies in several public-domain books, merges the counts, and plots the most common
words. Each book is counted in its own job, so the count step runs in parallel; a single merge job waits
for all of them. See `README.md` for the full pipeline description and options.

**Workflow Jobs:**
1. `count_words` - one per book: strip the Gutenberg header/footer, drop stopwords, count words
2. `merge_counts` - fan-in: sum every book's counts into `merged_counts.csv`
3. `plot_top_words` - bar chart of the top N words, stacked by book

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.9+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Input Data

Nothing to prepare: the books are fetched at run time. `create_replica_catalog()` registers each book's
Gutenberg URL (`https://www.gutenberg.org/cache/epub/<id>/pg<id>.txt`) as its physical file name, and
Pegasus's stage-in job downloads it with retries.

## 2. Build the Apptainer Container

The workflow's jobs run inside an Apptainer container (`Apptainer/Wordfreq_Container.def`) with Python and
matplotlib. Building it takes a minute or two; skip the rebuild if the `.sif` already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/Wordfreq_Container.sif ]; then
    apptainer build Apptainer/Wordfreq_Container.sif Apptainer/Wordfreq_Container.def
else
    echo "Container already built: Apptainer/Wordfreq_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `WordfreqWorkflow` class from `workflow_generator.py` — the same generator used
by the command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs) and
writes it to `workflow.yml`. Nothing about the workflow is defined here: change the pipeline in
`workflow_generator.py`, and both the notebook and the CLI pick it up.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import WordfreqWorkflow

DAGFILE = "workflow.yml"
EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

# The workflow-specific inputs the CLI would parse
BOOKS = ["11", "84", "1342", "1661", "2701"]  # Alice, Frankenstein, Pride and Prejudice, Sherlock Holmes, Moby Dick
ARGS = argparse.Namespace(top_n=20, min_length=3)

workflow = WordfreqWorkflow(dagfile=DAGFILE, books=BOOKS)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating workflow DAG...")
workflow.create_workflow(ARGS)

workflow.write()
print("\nWorkflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. You should see one `count_words`
node per book fanning in to `merge_counts`, followed by `plot_top_words`.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog
chosen above.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/`:

| Output | Description |
|--------|-------------|
| `merged_counts.csv` | Every word with its total count, number of books it appears in, and per-book counts |
| `top_words.png` | Bar chart of the top 20 words, stacked by book |

In [ ]:
!ls -ltR output/

In [ ]:
import csv

with open("output/merged_counts.csv") as fh:
    for i, row in enumerate(csv.reader(fh)):
        print(row)
        if i == 10:
            break

The top 20 words across all books:

In [ ]:
Image(filename="output/top_words.png")